Загрузите ZIP на Drive по пути ниже. Выберите T4 и выполните все ячейки.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
ARCHIVE = Path("/content/drive/MyDrive/nlp_hw2/inference_avg3_mid.zip")
PROJECT = Path("/content/hw2_inference")


In [ ]:
import hashlib
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
if not ARCHIVE.is_file():
    raise FileNotFoundError(str(ARCHIVE))
local_zip = Path("/content/inference_avg3_mid.zip")
shutil.copyfile(ARCHIVE, local_zip)
digest = hashlib.sha256()
with local_zip.open("rb") as stream:
    for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
        digest.update(chunk)
if digest.hexdigest() != "683828f0d4e2a35d55c09ce396312da5c3c7ef4dc096eff180492e4357ee350d":
    raise ValueError("ZIP отличается от подготовленного файла")
with zipfile.ZipFile(local_zip) as archive:
    archive.extractall(PROJECT)
sys.path.insert(0, str(PROJECT))


In [ ]:
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(PROJECT / "requirements.txt"),
], check=True)


In [ ]:
import time

started = time.perf_counter()
import torch
import transformers
import tokenizers
from inference import run_inference

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")
print("Torch:", torch.__version__, "Transformers:", transformers.__version__,
      "Tokenizers:", tokenizers.__version__)
submission, timing = run_inference(
    PROJECT / "weights/avg3_mid", PROJECT / "data", PROJECT / "outputs",
    started=started,
)
print(f"Генерация 1919 ответов: {timing['generation_seconds']:.2f} с")
print(f"Полная inference-ячейка: {timing['full_inference_seconds']:.2f} с")


In [ ]:
from google.colab import files

result = Path("/content/t4_results.zip")
with zipfile.ZipFile(result, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ("submission.jsonl", "timing.json"):
        archive.write(PROJECT / "outputs" / name, name)
files.download(str(result))
